# MeguvizCAC v0.5.0-ApexLoop A100-80GB 全量训练（超越 OjinjinNet 高标准版）

完整落地 `docs/roadmap-v0.5.0-research-plan.md` 与 `v0.4.0` R1/R2/R3 全部九条实战经验：
1. **Modern `mmBERT-small` 跨词表迁移 + `28k SuperBPE` 零 `[UNK]`**：彻底取代老旧 `bge-small-zh` 21k 词表，原生打通中/英/日/韩/代码与 `M04` 中英技术评论。
2. **端侧极速微架构（Phase 0 本地实测 `1.06 ms/row`）**：彻底淘汰 `mHC-4` 与 `CharStem`，采用 `1L RoPE+RMSNorm+GeGLU Prelude (带 Exit-0) + 32-d 零隐私 PIFE 文体 + 静态 16-Slot 工作台 + 1L Zero-LoRA InfiLoop/SCORE 连续进度 Core + L2 256-d 原型库`。
3. **四道反污染数据铁律**：保护桶白名单（严禁弱伪标污染 `structured_human_hard`）、非静默零泄漏审计（剔除全部 R2 陷阱撞车对并扩充多语/技术对）、`Gold Retention = 100%` 硬断言。
4. **四阶步数恒定闭环**：Stage-0 `100% RTD` 热启动 → Stage-1 `MuonClip` 全库蒸馏（持久存档 `v05_stage1_ckpt.pt`）→ Stage-2 固定 `2,400` 步 EMA+PIFE 对抗淬炼 → Stage-3 固定 `3,000` 步 `LoopQ` 全轨迹 QAT（覆盖 `R=2/4`）+ SWA。
5. **大样本平滑校准**：`n=2,400` 独立分层验证集 + 锐利温度烘焙（`T ∈ [0.9, 1.8]`）+ 分长度桶 Smooth PAVA + `Split Conformal 3-Way`（目标灰区 `<= 18%`）+ W4-Emb/W8-Body 边缘瘦身包（`<= 6.5 MB`）。


In [ ]:
# Cell 1: 挂载 Google Drive、更新仓库与依赖（若已复用 R3 的 A100-80GB 运行时，/content/data 已就绪无需重下！）
import os, torch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
REPO_DIR = '/content/MeguvizCAC'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} pull --ff-only
%cd /content/MeguvizCAC
!git log -3 --oneline
!pip -q install -U tokenizers onnx onnx_ir ml_dtypes onnxruntime transformers scipy scikit-learn huggingface_hub
assert Path('/content/data/v04-merged-1m4-r3.jsonl.gz').is_file(), '缺少 /content/data/v04-merged-1m4-r3.jsonl.gz，请先执行 R3 笔记本的 Cell 2 下载并清洗语料！'
print('V05_ENV_READY | GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB')


In [ ]:
# Cell 2: 启动后台抗断线 v0.5.0-ApexLoop A100-80GB 全量训练（自动 --probe-vram、双写 Drive、输出 15 项门禁并触发下载）
from meguviz_cac.colab_v05_launcher import run_v05_in_colab
run_v05_in_colab(poll_seconds=90)


In [ ]:
# Cell 3: 断线重连监控 / 手动补下载单元
import json
from pathlib import Path
from google.colab import files

done = Path('/content/v05_done.json')
if done.is_file():
    info = json.loads(done.read_text(encoding='utf-8'))
    print(json.dumps(info, ensure_ascii=False, indent=2))
    files.download(info['full_zip'])
else:
    !tail -n 30 /content/log_v05_driver.txt 2>/dev/null || true
    !tail -n 20 /content/log_v05_full.txt 2>/dev/null || true
